<a href="https://colab.research.google.com/github/sandroormeno/TensorFlow-LITE-ESP32-ARDUINO/blob/master/tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tensorflow==2.1.0

In [ ]:
%tensorflow_version 2.x

In [ ]:
from __future__ import absolute_import, division, print_function, unicode_literals
import pandas
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from keras.utils import to_categorical, np_utils
from sklearn.model_selection import train_test_split
from keras import backend as K

In [ ]:
print("TensorFlow version: {}".format(tf.__version__))

In [ ]:
url = "https://raw.githubusercontent.com/sandroormeno/Tensorflow-basic-examples/master/datos/iris.csv"
datosIris = pandas.read_csv(url)
datosIris.head()

In [ ]:
datos=datosIris.values[:,:4]
etiquetas=datosIris.values[:,4]

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(datos,etiquetas,train_size=0.8,random_state=23)

In [ ]:
def one_hot(arr):
    uniques, ids = np.unique(arr, return_inverse=True)
    return np_utils.to_categorical(ids, len(uniques))
y_train_ohe=one_hot(y_train)
y_test_ohe=one_hot(y_test)

In [ ]:
# los datos de entrada dan un error, al no poder convertir Float en tensores.
# soluci?n aqu? : https://blog.csdn.net/weixin_44473755/article/details/102939703
X_train_1 = K.cast_to_floatx (X_train)
X_test_1 = K.cast_to_floatx (X_test)

In [ ]:
print (X_train_1[20] )

In [ ]:
# El modelo
model= tf.keras.Sequential()

model.add(layers.Dense(16, activation='relu', input_dim=4))
model.add(layers.Dense(16, activation='relu'))
model.add(layers.Dense(16, activation='relu'))
model.add(layers.Dense(3, activation='sigmoid'))

# Configulaci?n del entrenamiento
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
model.fit(X_train_1,y_train_ohe, epochs=100,batch_size=1,verbose=1)

In [ ]:
loss, accuracy = model.evaluate(X_test_1, y_test_ohe, verbose=0)
print("Test accuracy : " + str(accuracy))

In [ ]:
# Stay here
# Convert the model to the TensorFlow Lite format without quantization
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.OPTIMIZE_FOR_SIZE]
tflite_model = converter.convert()

# Save the model to disk
open("exercice_model.tflite", "wb").write(tflite_model)

In [ ]:
X_test = X_test.astype('float32')

# Instantiate an interpreter for each model
exercice_model = tf.lite.Interpreter('exercice_model.tflite')

# Allocate memory for each model
exercice_model.allocate_tensors()

# Get the input and output tensors so we can feed in values and get the results
input_details = exercice_model.get_input_details()
output_details = exercice_model.get_output_details()

test = 15

input_data = X_test_1[test].reshape((1,4))
exercice_model.set_tensor(input_details[0]['index'],input_data )

exercice_model.invoke()
  
output_data = exercice_model.get_tensor(output_details[0]['index'])
    
  
print(output_data)  
print(y_test[test])

In [ ]:
import os
basic_model_size = os.path.getsize("exercice_model.tflite")
print("Basic model is %d bytes" % basic_model_size)

In [ ]:
# Install xxd if it is not available
!sudo apt-get update
!sudo apt-get install xxd

In [ ]:
# Save the file as a C source file
!xxd -i exercice_model.tflite > exercice_model.cc
# Print the source file
!cat exercice_model.cc

In [ ]:
for i in range(len(X_test_1)):
  salida = str(X_test_1[i]) + str(y_test_ohe[i])
  print(salida)

In [ ]:
print(X_test_1.shape)

In [ ]:
print(y_test_ohe)